In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
import lightgbm as lgb

# ----------------------------
# 1. Загрузка данных
# ----------------------------
train = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

# Сохраним id из test для сабмита
test_ids = test['id'].copy()

# Удалим 'id' из обучающих и тестовых данных
X = train.drop(columns=['id', 'Personality'])
y = train['Personality']
X_test = test.drop(columns=['id'])

# ----------------------------
# 2. Определение типов колонок
# ----------------------------
# Числовые колонки — те, где dtype числовой
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# ----------------------------
# 3. Заполнение пропусков
# ----------------------------
# Числовые: среднее
for col in num_cols:
    mean_val = X[col].mean()
    X[col].fillna(mean_val, inplace=True)
    X_test[col].fillna(mean_val, inplace=True)

# Категориальные: заполним пропуски как "missing"
for col in cat_cols:
    X[col].fillna('missing', inplace=True)
    X_test[col].fillna('missing', inplace=True)

# ----------------------------
# 4. One-Hot Encoding
# ----------------------------
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat_encoded = encoder.fit_transform(X[cat_cols])
X_test_cat_encoded = encoder.transform(X_test[cat_cols])

# Преобразуем в DataFrame с правильными именами колонок
cat_feature_names = encoder.get_feature_names_out(cat_cols)
X_cat_df = pd.DataFrame(X_cat_encoded, columns=cat_feature_names, index=X.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=cat_feature_names, index=X_test.index)

# Объединяем числовые и закодированные признаки
X_final = pd.concat([X[num_cols], X_cat_df], axis=1)
X_test_final = pd.concat([X_test[num_cols], X_test_cat_df], axis=1)

# ----------------------------
# 5. LightGBM + 5-Fold OOF
# ----------------------------
n_folds = 50
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_final))
test_preds_proba = np.zeros(len(X_test_final))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_final, y)):
    print(f"Fold {fold + 1}/{n_folds}")
    
    X_train_fold, X_val_fold = X_final.iloc[train_idx], X_final.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(
        objective='binary',
        metric='binary_logloss',
        boosting_type='gbdt',
        num_leaves=31,
        learning_rate=0.05,
        feature_fraction=0.9,
        bagging_fraction=0.8,
        bagging_freq=5,
        verbose=-1,
        random_state=42
    )
    
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)]
    )
    
    # OOF предсказания (вероятности класса 1)
    oof_preds[val_idx] = model.predict_proba(X_val_fold)[:, 1]
    
    # Предсказания на тесте
    test_preds_proba += model.predict_proba(X_test_final)[:, 1] / n_folds

# Оценка качества (необязательно, но полезно)
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y, oof_preds)
print(f"\nOOF ROC AUC: {auc:.5f}")

# ----------------------------
# 6. Создание сабмита
# ----------------------------
# Бинарные предсказания: порог 0.5
test_pred_class = (test_preds_proba > 0.5).astype(int)

# Преобразуем 0 → 'introvert', 1 → 'extrovert'
test_pred_labels = np.where(test_pred_class == 1, 'Extrovert', 'Introvert')

# Формируем DataFrame
submission = pd.DataFrame({
    'id': test_ids,
    'Personality': test_pred_labels
})

# Сохраняем
submission.to_csv('submission2.csv', index=False)
print("\nФайл submission.csv успешно создан!")

/tmp/ipykernel_55/590008418.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col].fillna(mean_val, inplace=True)
/tmp/ipykernel_55/590008418.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.metho

Fold 1/50
Fold 2/50
Fold 3/50
Fold 4/50
Fold 5/50
Fold 6/50
Fold 7/50
Fold 8/50
Fold 9/50
Fold 10/50
Fold 11/50
Fold 12/50
Fold 13/50
Fold 14/50
Fold 15/50
Fold 16/50
Fold 17/50
Fold 18/50
Fold 19/50
Fold 20/50
Fold 21/50
Fold 22/50
Fold 23/50
Fold 24/50
Fold 25/50
Fold 26/50
Fold 27/50
Fold 28/50
Fold 29/50
Fold 30/50
Fold 31/50
Fold 32/50
Fold 33/50
Fold 34/50
Fold 35/50
Fold 36/50
Fold 37/50
Fold 38/50
Fold 39/50
Fold 40/50
Fold 41/50
Fold 42/50
Fold 43/50
Fold 44/50
Fold 45/50
Fold 46/50
Fold 47/50
Fold 48/50
Fold 49/50
Fold 50/50

OOF ROC AUC: 0.96961

Файл submission.csv успешно создан!


In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import lightgbm as lgb

# Загрузка данных
train = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

# Сохраняем id для сабмита
test_ids = test['id'].copy()

# Удаляем 'id'
X = train.drop(columns=['id', 'Personality'])
y = train['Personality']
X_test = test.drop(columns=['id'])

# Определяем числовые и категориальные колонки
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Заполняем пропуски
# Заполняем пропуски — БЕЗ inplace
for col in num_cols:
    mean_val = X[col].mean()
    X[col] = X[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)

for col in cat_cols:
    X[col] = X[col].fillna('missing')
    X_test[col] = X_test[col].fillna('missing')
    
# One-Hot Encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat = encoder.fit_transform(X[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

# Имена новых колонок
cat_feature_names = encoder.get_feature_names_out(cat_cols)
X_cat_df = pd.DataFrame(X_cat, columns=cat_feature_names, index=X.index)
X_test_cat_df = pd.DataFrame(X_test_cat, columns=cat_feature_names, index=X_test.index)

# Объединяем
X_final = pd.concat([X[num_cols], X_cat_df], axis=1)
X_test_final = pd.concat([X_test[num_cols], X_test_cat_df], axis=1)

# Обучаем LightGBM на всех данных
model = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    boosting_type='gbdt',
    num_leaves=31,
    learning_rate=0.05,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42
)

model.fit(X_final, y)

# Предсказываем классы (0 или 1)
preds = model.predict(X_test_final)

# Преобразуем в метки
pred_labels = np.where(preds == 1, 'Extrovert', 'Introvert')

# Сохраняем сабмит
submission = pd.DataFrame({'id': test_ids, 'Personality': pred_labels})
submission.to_csv('submission3.csv', index=False)

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# 1. Загрузка данных
# ----------------------------
train = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')
test_ids = test['id'].copy()

X = train.drop(columns=['id', 'Personality'])
y = train['Personality']
X_test = test.drop(columns=['id'])

# ----------------------------
# 2. Обработка категориальных признаков
# ----------------------------
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Заполнение пропусков
for col in num_cols:
    mean_val = X[col].mean()
    X[col].fillna(mean_val, inplace=True)
    X_test[col].fillna(mean_val, inplace=True)

for col in cat_cols:
    X[col].fillna('missing', inplace=True)
    X_test[col].fillna('missing', inplace=True)

# Кодирование категорий для XGBoost (LightGBM и CatBoost работают с текстом)
label_encoders = {}
X_encoded = X.copy()
X_test_encoded = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    # Объединяем значения для корректного кодирования
    all_vals = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(all_vals)
    X_encoded[col] = le.transform(X[col].astype(str))
    X_test_encoded[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# ----------------------------
# 3. Параметры моделей
# ----------------------------
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': 0
}

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

cat_params = {
    'objective': 'Logloss',
    'eval_metric': 'Logloss',
    'depth': 6,
    'learning_rate': 0.05,
    'random_seed': 42,
    'verbose': 0
}

# ----------------------------
# 4. Обучение и OOF-предсказания
# ----------------------------
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

preds_xgb = np.zeros(len(X_test))
preds_lgb = np.zeros(len(X_test))
preds_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold + 1}/{n_folds}")
    
    X_train, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    X_train_lgb = X.iloc[train_idx]
    X_val_lgb = X.iloc[val_idx]

    # --- XGBoost ---
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    preds_xgb += model_xgb.predict_proba(X_test_encoded)[:, 1] / n_folds

    # --- LightGBM ---
    train_data = lgb.Dataset(X_train_lgb, label=y_train, categorical_feature=cat_cols)
    val_data = lgb.Dataset(X_val_lgb, label=y_val, categorical_feature=cat_cols, reference=train_data)
    model_lgb = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=10000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    oof_lgb[val_idx] = model_lgb.predict(X_val_lgb)
    preds_lgb += model_lgb.predict(X_test) / n_folds

    # --- CatBoost ---
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(
        X_train_lgb, y_train,
        eval_set=(X_val_lgb, y_val),
        cat_features=cat_cols,
        early_stopping_rounds=100,
        verbose=0
    )
    oof_cat[val_idx] = model_cat.predict_proba(X_val_lgb)[:, 1]
    preds_cat += model_cat.predict_proba(X_test)[:, 1] / n_folds

# ----------------------------
# 5. Оценка качества
# ----------------------------
auc_xgb = roc_auc_score(y, oof_xgb)
auc_lgb = roc_auc_score(y, oof_lgb)
auc_cat = roc_auc_score(y, oof_cat)

print(f"\nOOF AUC — XGBoost: {auc_xgb:.5f}, LightGBM: {auc_lgb:.5f}, CatBoost: {auc_cat:.5f}")

# ----------------------------
# 6. Финальный ансамбль (взвешенное среднее)
# ----------------------------
# Пример: равные веса (можно оптимизировать)
weights = [1/3, 1/3, 1/3]
final_pred = weights[0] * preds_xgb + weights[1] * preds_lgb + weights[2] * preds_cat

# Бинарный класс (если нужно)
# final_class = (final_pred > 0.5).astype(int)

# ----------------------------
# 7. Сохранение сабмита
# ----------------------------
submission = pd.DataFrame({
    'id': test_ids,
    'Personality': final_class  # или final_class, если нужен класс
})
submission.to_csv('ensemble_submission.csv', index=False)
print("\nФайл ensemble_submission.csv успешно создан!")

Fold 1/5


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# ----------------------------
# 1. Загрузка данных
# ----------------------------
train = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')
test_ids = test['id'].copy()

X = train.drop(columns=['id', 'Personality'])
y = train['Personality']
X_test = test.drop(columns=['id'])

# ----------------------------
# 2. Обработка категориальных признаков
# ----------------------------
# После загрузки X и y
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Заполнение пропусков
for col in num_cols:
    mean_val = X[col].mean()
    X[col] = X[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)

for col in cat_cols:
    X[col] = X[col].fillna('missing')
    X_test[col] = X_test[col].fillna('missing')

# === КОДИРОВАНИЕ КАТЕГОРИЙ В ЦЕЛЫЕ ЧИСЛА ДЛЯ ВСЕХ МОДЕЛЕЙ ===
label_encoders = {}
X_encoded = X.copy()
X_test_encoded = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    # Обучаем на объединённых данных (train + test)
    all_vals = pd.concat([X[col], X_test[col]], axis=0)
    le.fit(all_vals.astype(str))
    X_encoded[col] = le.transform(X[col].astype(str))
    X_test_encoded[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Теперь X_encoded и X_test_encoded содержат ТОЛЬКО числовые колонки

# ----------------------------
# 3. Обучение моделей на всём датасете
# ----------------------------

# XGBoost
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
#xgb_model.fit(X_encoded, y)

# LightGBM
lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
#lgb_model.fit(X, y, categorical_feature=cat_cols)

# CatBoost
cat_model = CatBoostClassifier(
    objective='Logloss',
    eval_metric='Logloss',
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=0
)
#cat_model.fit(X, y, cat_features=cat_cols)
# После загрузки y (до обучения моделей)
print("Уникальные значения y:", y.unique())

# Кодируем метки: 'Introvert' → 0, 'Extrovert' → 1
label_map = {'Introvert': 0, 'Extrovert': 1}
y_encoded = y.map(label_map)

# Убедитесь, что все значения корректны
if y_encoded.isna().any():
    raise ValueError("Обнаружены неизвестные метки в y!")

# Кодируем целевую переменную
y_encoded = y.map({'Introvert': 0, 'Extrovert': 1})
if y_encoded.isna().any():
    raise ValueError("Неизвестные метки в y!")

# XGBoost — работает с числами

xgb_model.fit(X_encoded, y_encoded)

# LightGBM — принимает числовые категории

lgb_model.fit(X_encoded, y_encoded, categorical_feature=cat_cols)  # OK, если cat_cols — числа

# CatBoost — может работать со строками, но мы уже закодировали → тоже числа
# Но CatBoost требует указания cat_features как индексы или имена

cat_model.fit(X_encoded, y_encoded, cat_features=cat_cols, verbose=0)
# ----------------------------
# 4. Предсказания на тесте
# ----------------------------
# Предсказания на тесте — ВСЕГДА на закодированных данных!
pred_xgb = xgb_model.predict_proba(X_test_encoded)[:, 1]
pred_lgb = lgb_model.predict_proba(X_test_encoded)[:, 1]
pred_cat = cat_model.predict_proba(X_test_encoded)[:, 1]

# Ансамбль
final_pred = (pred_xgb + pred_lgb + pred_cat) / 3.0
final_class = (final_pred > 0.5).astype(int)

# Обратное преобразование меток
final_labels = np.where(final_class == 1, 'Extrovert', 'Introvert')

# ----------------------------
# 5. Сохранение сабмита
# ----------------------------


submission = pd.DataFrame({
    'id': test_ids,
    'Personality': final_labels  # <-- строковые метки
})
submission.to_csv('ensemble_submission.csv', index=False)
print("Файл ensemble_submission.csv успешно создан!")

Уникальные значения y: ['Extrovert' 'Introvert']
Файл ensemble_submission.csv успешно создан!
